# 🧠 Lab 01 — AI Foundations
**ML vs Deep Learning vs GenAI: hands-on comparison**

---
**Real-world scenario:** ASML in Veldhoven builds the world's only EUV lithography machines.
Each wafer goes through hundreds of inspection steps. Catching defects at nanometre scale
matters enormously — a single missed defect can ruin a €30 000 wafer.
In this lab you will train the same style of models ASML uses: first a classical ML classifier,
then a neural network, and compare accuracy vs interpretability.

**What you will build:**
1. Train a decision tree and a neural network on the same dataset
2. Visualise how a neural network learns over epochs
3. Compare accuracy vs interpretability trade-offs

**Estimated time:** 45 min  |  **Level:** Beginner

In [ ]:
%pip install -q numpy scikit-learn matplotlib torch torchvision

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, classification_report
import torch
import torch.nn as nn

np.random.seed(42)
torch.manual_seed(42)
print('All imports OK')

## Part 1 — Create a synthetic wafer defect dataset
We simulate a binary classification problem: defective wafer (1) vs clean wafer (0).
Features represent sensor readings: edge intensity, roughness, reflectance variance, etc.

In [ ]:
X, y = make_classification(
    n_samples=2000,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    n_clusters_per_class=2,
    class_sep=1.2,
    random_state=42
)

feature_names = [
    'edge_intensity', 'roughness', 'reflectance_var', 'pattern_score',
    'cd_error', 'overlay_x', 'overlay_y', 'focus_error',
    'resist_thickness', 'dose_error', 'noise_level', 'thermal_drift'
]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Defect rate: {y.mean():.1%}')

## Part 2 — Classical ML: Decision Tree
A decision tree is fully interpretable — you can print the rules and explain every decision.
This matters in manufacturing: ASML engineers need to know *why* a wafer was flagged.

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
dt_preds = dt.predict(X_test)
dt_acc = accuracy_score(y_test, dt_preds)

print(f'Decision Tree accuracy: {dt_acc:.3f}')
print()
print(classification_report(y_test, dt_preds, target_names=['Clean', 'Defective']))

In [ ]:
# Print the first 3 levels of the tree — this is what interpretability looks like
tree_rules = export_text(dt, feature_names=feature_names, max_depth=3)
print(tree_rules)

## Part 3 — Deep Learning: Neural Network
A neural network learns complex non-linear boundaries. Higher accuracy, but a black box.

In [ ]:
class WaferNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(12, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x).squeeze(1)

X_tr = torch.FloatTensor(X_train)
y_tr = torch.FloatTensor(y_train)
X_te = torch.FloatTensor(X_test)
y_te = torch.FloatTensor(y_test)

model = WaferNet()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.BCELoss()

train_losses, test_accs = [], []

for epoch in range(80):
    model.train()
    opt.zero_grad()
    preds = model(X_tr)
    loss = loss_fn(preds, y_tr)
    loss.backward()
    opt.step()
    train_losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        te_preds = (model(X_te) > 0.5).float()
        acc = (te_preds == y_te).float().mean().item()
        test_accs.append(acc)

nn_acc = test_accs[-1]
print(f'Neural Network accuracy: {nn_acc:.3f}')

## Part 4 — Visualise learning over epochs

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses, color='#E8A020', linewidth=2)
ax1.set_title('Training Loss over Epochs', fontsize=13)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('BCE Loss')
ax1.grid(alpha=0.3)

ax2.plot(test_accs, color='#2EC4B6', linewidth=2)
ax2.axhline(dt_acc, color='#F77F00', linestyle='--', linewidth=1.5, label=f'Decision Tree ({dt_acc:.3f})')
ax2.set_title('Test Accuracy over Epochs', fontsize=13)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)

plt.suptitle('Neural Network vs Decision Tree — Wafer Defect Detection', fontsize=14)
plt.tight_layout()
plt.show()

## Part 5 — Accuracy vs Interpretability trade-off

In [ ]:
models = ['Decision Tree', 'Neural Network']
accuracies = [dt_acc, nn_acc]
interpretability = [0.95, 0.05]  # subjective score 0-1

fig, ax = plt.subplots(figsize=(7, 5))
colors = ['#F77F00', '#2EC4B6']
for i, (m, a, interp) in enumerate(zip(models, accuracies, interpretability)):
    ax.scatter(interp, a, s=200, color=colors[i], zorder=5, label=m)
    ax.annotate(f'{m}\nacc={a:.3f}', (interp, a),
                textcoords='offset points', xytext=(10, 5), fontsize=10)

ax.set_xlabel('Interpretability (1 = fully explainable)', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('The Accuracy–Interpretability Trade-off\n(core tension in applied ML)', fontsize=12)
ax.set_xlim(-0.05, 1.05)
ax.set_ylim(0.7, 1.0)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('\n--- Summary ---')
print(f'Decision Tree:  acc={dt_acc:.3f}, interpretable=YES, rules=printable')
print(f'Neural Network: acc={nn_acc:.3f}, interpretable=NO,  rules=millions of weights')
print('\nASML context: in regulated manufacturing, you often need BOTH — use NNs for performance')
print('and techniques like SHAP / LIME to explain individual predictions post-hoc.')

## ✅ Lab Complete
You have:
- Trained a decision tree (interpretable, fast, good baseline)
- Trained a neural network (higher accuracy, black box)
- Visualised learning curves and the accuracy–interpretability trade-off

**Next:** Lab 02 — Generative AI & LLMs (tokenisation, sampling, prompt engineering)